In [1]:
from datasets import load_dataset

ds = load_dataset("scikit-fingerprints/MoleculeNet_Tox21")
tox21 = ds["train"].to_pandas()

print(tox21.shape)
print(tox21.head())

(7831, 13)
                                              SMILES  NR-AR  NR-AR-LBD  \
0                       CCOc1ccc2nc(S(N)(=O)=O)sc2c1    0.0        0.0   
1                          CCN1C(=O)NC(c2ccccc2)C1=O    0.0        0.0   
2  CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...    NaN        NaN   
3                    CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C    0.0        0.0   
4                          CC(O)(P(=O)(O)O)P(=O)(O)O    0.0        0.0   

   NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD  NR-PPAR-gamma  SR-ARE  SR-ATAD5  \
0     1.0           NaN    NaN        0.0            0.0     1.0       0.0   
1     0.0           0.0    0.0        0.0            0.0     NaN       0.0   
2     NaN           NaN    NaN        NaN            NaN     0.0       NaN   
3     0.0           0.0    0.0        0.0            0.0     NaN       0.0   
4     0.0           0.0    0.0        0.0            0.0     0.0       0.0   

   SR-HSE  SR-MMP  SR-p53  
0     0.0     0.0     0.0  
1     NaN     0.0  

In [2]:
# 1. SMILES 파싱 성공률 체크
from rdkit import Chem
tox21["mol"] = tox21["SMILES"].apply(Chem.MolFromSmiles)
success_rate = tox21["mol"].notna().mean()
print(f"파싱 성공률: {success_rate*100:.1f}%")

# 2. 컬럼별 결측치 개수
print(tox21.drop(columns=["mol"]).isna().sum())

파싱 성공률: 100.0%
SMILES              0
NR-AR             566
NR-AR-LBD        1073
NR-AhR           1282
NR-Aromatase     2010
NR-ER            1638
NR-ER-LBD         876
NR-PPAR-gamma    1381
SR-ARE           1999
SR-ATAD5          759
SR-HSE           1364
SR-MMP           2021
SR-p53           1057
dtype: int64


In [3]:
for col in ["NR-AR", "NR-AhR", "SR-MMP"]:  # 결측 많은/적은 것 섞어서 확인
    print(f"\n{col}:")
    print(tox21[col].value_counts(dropna=True))


NR-AR:
NR-AR
0.0    6956
1.0     309
Name: count, dtype: int64

NR-AhR:
NR-AhR
0.0    5781
1.0     768
Name: count, dtype: int64

SR-MMP:
SR-MMP
0.0    4892
1.0     918
Name: count, dtype: int64


In [4]:
summary = []
for col in tox21.columns[1:13]:  # SMILES 제외 12개 라벨
    vc = tox21[col].value_counts(dropna=True)
    missing = tox21[col].isna().sum()
    pos_ratio = vc.get(1.0, 0) / vc.sum() * 100 if vc.sum() > 0 else 0
    summary.append({"task": col, "missing": missing, "pos_ratio(%)": round(pos_ratio, 1)})

import pandas as pd
print(pd.DataFrame(summary).sort_values("pos_ratio(%)"))

             task  missing  pos_ratio(%)
6   NR-PPAR-gamma     1381           2.9
1       NR-AR-LBD     1073           3.5
8        SR-ATAD5      759           3.7
0           NR-AR      566           4.3
5       NR-ER-LBD      876           5.0
3    NR-Aromatase     2010           5.2
9          SR-HSE     1364           5.8
11         SR-p53     1057           6.2
2          NR-AhR     1282          11.7
4           NR-ER     1638          12.8
10         SR-MMP     2021          15.8
7          SR-ARE     1999          16.2


In [5]:
# 컬럼 목록만 확인 (저장은 InChIKey 계산 이후 셀에서 한 번에 수행)
print(tox21.columns.tolist())

['SMILES', 'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'mol']


In [6]:
# 각 CSV에 InChIKey 컬럼 추가
tox21["InChIKey"] = tox21["mol"].apply(
    lambda m: Chem.MolToInchiKey(m) if m is not None else None
)
print(tox21[["SMILES", "InChIKey"]].head(2))

                         SMILES                     InChIKey
0  CCOc1ccc2nc(S(N)(=O)=O)sc2c1  OUZWUKMCLIBBOG-UHFFFAOYSA-N
1     CCN1C(=O)NC(c2ccccc2)C1=O  SZQIFWWUIBRPBZ-UHFFFAOYSA-N


In [7]:
tox21.info()

<class 'pandas.DataFrame'>
RangeIndex: 7831 entries, 0 to 7830
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   SMILES         7831 non-null   str    
 1   NR-AR          7265 non-null   float64
 2   NR-AR-LBD      6758 non-null   float64
 3   NR-AhR         6549 non-null   float64
 4   NR-Aromatase   5821 non-null   float64
 5   NR-ER          6193 non-null   float64
 6   NR-ER-LBD      6955 non-null   float64
 7   NR-PPAR-gamma  6450 non-null   float64
 8   SR-ARE         5832 non-null   float64
 9   SR-ATAD5       7072 non-null   float64
 10  SR-HSE         6467 non-null   float64
 11  SR-MMP         5810 non-null   float64
 12  SR-p53         6774 non-null   float64
 13  mol            7831 non-null   object 
 14  InChIKey       7831 non-null   str    
dtypes: float64(12), object(1), str(2)
memory usage: 1.4+ MB


In [8]:
# mol 컬럼은 제거하고, InChIKey는 포함해서 저장
tox21_to_save = tox21.drop(columns=["mol"])
tox21_to_save.to_csv("./data/hazard/moleculenet_tox21.csv", index=False)
print("저장 완료:", tox21_to_save.shape)

# 저장된 파일이 InChIKey를 포함해서 제대로 저장됐는지 재확인
check = pd.read_csv("./data/hazard/moleculenet_tox21.csv")
print(check.shape)
print("InChIKey 컬럼 있음?:", "InChIKey" in check.columns)
print(check[["SMILES", "InChIKey"]].head(2))

저장 완료: (7831, 14)
(7831, 14)
InChIKey 컬럼 있음?: True
                         SMILES                     InChIKey
0  CCOc1ccc2nc(S(N)(=O)=O)sc2c1  OUZWUKMCLIBBOG-UHFFFAOYSA-N
1     CCN1C(=O)NC(c2ccccc2)C1=O  SZQIFWWUIBRPBZ-UHFFFAOYSA-N
